# Lab 3 — Tokenization, Stemming, Lemmatization and WordNet
**Project:** French Environmental Text Intelligence (Group 3, French)

**What this notebook does, in plain words.** A computer sees *system* and *systems*, or *pollution* and *polluants*, as completely different words. To search or classify text well, we map the different forms of a word to one shared form:
- **Stemming** cuts the endings off with rules (*connected → connect*, *argued → argu*). It is fast but crude: the result may not be a real word, and unrelated words can get the same stem.
- **Lemmatization** gives the dictionary form (*were → be*, *mice → mouse*). It needs the part of speech and a dictionary, so it is slower but more correct.
- **WordNet** is a dictionary of *meanings*: synonyms, "is-a" relations (*a car is a motor vehicle*) and similarity between concepts.

Parts 1–5 answer the Lab 3 sheet (in English, as given). **Part 6 applies the same ideas to our French corpus**, because this is what our search engine and classifier will need.

## Setup

In [1]:
# !pip install nltk spacy beautifulsoup4 requests
# !python -m spacy download en_core_web_md
# !python -m spacy download fr_core_news_md
import re, time
from collections import Counter, defaultdict
from itertools import combinations
import pandas as pd
import nltk
for pkg in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'stopwords']:
    nltk.download(pkg, quiet=True)
try:
    nltk.download('omw-2.0', quiet=True)       # newer NLTK versions use this name for the multilingual WordNet
except Exception:
    pass
import spacy
from spacy.language import Language
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet as wn

nlp = spacy.load('en_core_web_md')
print('spaCy', spacy.__version__, '| NLTK', nltk.__version__)

[nltk_data] Error loading punkt: Security Violation [pathsec.urlopen]:
[nltk_data]     refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.
[nltk_data] Error loading punkt_tab: Security Violation
[nltk_data]     [pathsec.urlopen]: refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be S

spaCy 3.8.18 | NLTK 3.10.4


---
# Part 1 — Tokenization, segmentation and subwords
## Question 1

In [2]:
text = "Jean-Claude, can you help please ?, . . . , I will give it you back ! Never forget . . . Thank you"
doc = nlp(text)
print('a) Sentences:')
for s in doc.sents:
    print('   ', repr(s.text))
print('\n   Tokens and lemmas:')
print(pd.DataFrame([(t.i, t.text, t.lemma_, t.pos_) for t in doc], columns=['i', 'token', 'lemma', 'POS']).T.to_string(header=False))

a) Sentences:
    'Jean-Claude, can you help please ?, . .'
    '.'
    ', I will give it you back !'
    'Never forget . . .'
    'Thank you'

   Tokens and lemmas:
i          0      1       2      3    4     5     6       7      8      9     10     11     12     13    14    15    16    17    18    19     20     21      22     23     24     25     26    27
token   Jean      -  Claude      ,  can   you  help  please      ?      ,      .      .      .      ,     I  will  give    it   you  back      !  Never  forget      .      .      .  Thank   you
lemma   Jean      -  Claude      ,  can   you  help  please      ?      ,      .      .      .      ,     I  will  give    it   you  back      !  never  forget      .      .      .  thank   you
POS    PROPN  PUNCT   PROPN  PUNCT  AUX  PRON  VERB    VERB  PUNCT  PUNCT  PUNCT  PUNCT  PUNCT  PUNCT  PRON   AUX  VERB  PRON  PRON   ADV  PUNCT    ADV    VERB  PUNCT  PUNCT  PUNCT   VERB  PRON


**b) A custom delimiter.** spaCy cuts *inside* the spaced ellipsis `. . .` (see a): `'. .'` and `'.'` end up in different sentences). We add a pipeline component, placed *before* the parser, with two rules: the dots of an ellipsis (`...`, `…` or `. . .`) stay in the same sentence, and the token *after* the ellipsis starts a new sentence.

In [3]:
@Language.component('ellipsis_delimiter')
def ellipsis_delimiter(doc):
    i = 0
    while i < len(doc):
        if doc[i].text in ('...', '…'):
            end = i + 1
        elif doc[i:i + 3].text.replace(' ', '') == '...':      # the spaced form . . .
            end = i + 3
        else:
            i += 1
            continue
        for k in range(i + 1, end):
            doc[k].is_sent_start = False                       # keep the dots together
        if end < len(doc):
            doc[end].is_sent_start = True                      # the next token starts a sentence
        i = end
    return doc

nlp_custom = spacy.load('en_core_web_md')
nlp_custom.add_pipe('ellipsis_delimiter', before='parser')

for name, model in [('without customization', nlp), ('with the ellipsis delimiter', nlp_custom)]:
    sents = list(model(text).sents)
    print(f'c) {name}: {len(sents)} sentences')
    for s in sents:
        print('      ', repr(s.text))

c) without customization: 5 sentences
       'Jean-Claude, can you help please ?, . .'
       '.'
       ', I will give it you back !'
       'Never forget . . .'
       'Thank you'
c) with the ellipsis delimiter: 4 sentences
       'Jean-Claude, can you help please ?, . . .'
       ', I will give it you back !'
       'Never forget . . .'
       'Thank you'


In [4]:
# d) Three tokenizers
toks = {'split()': text.split(), 'nltk.word_tokenize': word_tokenize(text), 'spaCy': [t.text for t in doc]}
print(pd.DataFrame({'tokens': {k: len(v) for k, v in toks.items()}}))
for k, v in toks.items():
    print(f'{k:20s}', v)
# e) NLTK sentence segmentation
print('\ne) nltk.sent_tokenize:', len(sent_tokenize(text)), 'sentences')
for s in sent_tokenize(text):
    print('      ', repr(s))

                    tokens
split()                 24
nltk.word_tokenize      26
spaCy                   28
split()              ['Jean-Claude,', 'can', 'you', 'help', 'please', '?,', '.', '.', '.', ',', 'I', 'will', 'give', 'it', 'you', 'back', '!', 'Never', 'forget', '.', '.', '.', 'Thank', 'you']
nltk.word_tokenize   ['Jean-Claude', ',', 'can', 'you', 'help', 'please', '?', ',', '.', '.', '.', ',', 'I', 'will', 'give', 'it', 'you', 'back', '!', 'Never', 'forget', '.', '.', '.', 'Thank', 'you']
spaCy                ['Jean', '-', 'Claude', ',', 'can', 'you', 'help', 'please', '?', ',', '.', '.', '.', ',', 'I', 'will', 'give', 'it', 'you', 'back', '!', 'Never', 'forget', '.', '.', '.', 'Thank', 'you']

e) nltk.sent_tokenize: 8 sentences
       'Jean-Claude, can you help please ?, .'
       '.'
       '.'
       ', I will give it you back !'
       'Never forget .'
       '.'
       '.'
       'Thank you'


**Answers to question 1.**
- **a)** spaCy (English model) finds **5 sentences**. It cuts *inside* the spaced ellipsis: *"…please ?, . ."* and then a sentence that is just *"."*. The lemmas are the lower-cased words (*Never → never*, *Thank → thank*). There is nothing to inflect in this text.
- **b) / c)** Without customization: **5 sentences**. With the ellipsis delimiter: **4 sentences**: *Jean-Claude, can you help please ?, . . .* | *, I will give it you back !* | *Never forget . . .* | *Thank you*. That is what a human reader expects.
- **d)**

| Method | Tokens | *Jean-Claude* | *?* | *. . .* |
|---|---|---|---|---|
| `split()` | 24 | one token, comma glued (*Jean-Claude,*) | glued to the comma (*?,*) | three separate *.* (because of the spaces) |
| `nltk.word_tokenize` | 26 | one token | separate | three separate *.* |
| spaCy | 28 | **three tokens**: *Jean \| - \| Claude* (the English rules split hyphens) | separate | three separate *.* |

  None of the three recognises the spaced *. . .* as one ellipsis. They would if it were written *...* without spaces.
- **e)** `nltk.sent_tokenize` finds **8 sentences**, the worst result: Punkt treats every spaced *.* as a sentence end, so *"."* appears alone three times. Ranking: our custom spaCy (4) > spaCy default (5) > NLTK (8).

## Question 2 — Subwords (WordPiece)
Needs `pip install transformers` and an internet connection (the tokenizer is downloaded from Hugging Face). **Run on your laptop**; skipped automatically otherwise.

In [5]:
try:
    from transformers import AutoTokenizer
    wp = AutoTokenizer.from_pretrained('bert-base-uncased')
    s = 'Jean-Claude, can you help please?'
    print('a)', wp.tokenize(s))
    for w in ['unhappiness', 'organization', 'tokenization', 'retrieval', 'universiteti']:
        print('b)', f'{w:14s}', wp.tokenize(w))
    print('c) spaCy word tokens:', len(nlp(s)), '| WordPiece subword tokens:', len(wp.tokenize(s)))
except Exception as e:
    print('Skipped (needs transformers + internet):', type(e).__name__, str(e)[:120])

Skipped (needs transformers + internet): ModuleNotFoundError No module named 'transformers'


**What to look for when you run it:**
- **b)** Frequent English words like *organization* stay whole. Rarer ones are split into pieces that start with `##` (a piece that continues a word), e.g. *token + ##ization*. The Albanian word *universiteti* is not in BERT's English vocabulary, so it is cut into several pieces (something like *university*-like pieces plus *##ti*).
- **c)** On a normal English sentence the counts are close. The difference grows with rare words.
- **Why can subwords handle a word never seen in training?** Every word can be written as a sequence of known pieces, down to single characters. There is no "unknown word" (`<UNK>`): a new word is just a new combination of known pieces.

---
# Part 2 — A real text: the Guardian article
**3 a)** Download the page with `requests` and extract the paragraphs with BeautifulSoup. This needs an internet connection (**run on your laptop**). We do not store the article in the repository, because it is copyrighted.

If the download fails, the same analysis runs on the **news texts of our own French corpus** (with the French model), so the code always produces a result.

In [6]:
URL = 'https://www.theguardian.com/us-news/2022/jun/18/donald-trump-election-defense-fundraising-defrauded-fans'
try:
    import requests
    from bs4 import BeautifulSoup
    html = requests.get(URL, timeout=20, headers={'User-Agent': 'Mozilla/5.0 (student lab)'}).text
    soup = BeautifulSoup(html, 'html.parser')
    article_text = '\n'.join(p.get_text(' ', strip=True) for p in soup.select('article p') or soup.find_all('p'))
    assert len(article_text) > 1000
    nlp_art, SOURCE = nlp, 'the Guardian article (English)'
except Exception as e:
    print('Guardian not reachable here (', type(e).__name__, ') -> using the news texts of our French corpus instead.')
    from nlp_utils import load_corpus
    article_text = '\n'.join(d['text'] for d in load_corpus(kinds=['news']))
    nlp_art, SOURCE = spacy.load('fr_core_news_md'), 'the 16 news texts of our French corpus'
print('Text used:', SOURCE)
print(article_text[:500])

Guardian not reachable here ( ProxyError ) -> using the news texts of our French corpus instead.


Text used: the 16 news texts of our French corpus
Publié le 13 décembre 2020 
 
Le gouvernement canadien a déposé son « plan climatique renforcé ». L'objectif annoncé est d'atteindre et même de dépasser les objectifs de réduction des gaz à effet de serre (GES) que le gouvernement précédent du premier ministre conservateur Stephen Harper avait présentées en 2015 dans la foulée de l'Accord de Paris. La nouvelle cible du gouvernement du premier ministre Justin Trudeau prévoit des réductions de GES se situant entre 32 et 40 % d'ici 2030, par rappor


In [7]:
# b) a spaCy doc, c) tokens, sentences, average sentence length, longest sentence
art = nlp_art(article_text)
sents = list(art.sents)
print(f'{len(art)} tokens, {len(sents)} sentences, {len(art) / len(sents):.1f} tokens per sentence on average')
print('Longest sentence:', max(sents, key=len).text[:400])

6204 tokens, 198 sentences, 31.3 tokens per sentence on average
Longest sentence: Ce scénario implique d'installer des infrastructures renouvelables (éoliennes, panneaux solaires) même si le rythme d'installation devra être nettement plus élevé qu'aujourd'hui et le coût de ce système sera 10 milliards d'euros inférieur au scénario de référence sur l'ensemble du système (production, flexibilités et réseau).
À l'inverse, le scénario « réindustrialisation profonde » coûte 10 milli


In [8]:
# d) 10 most frequent words and lemmas (lower-case, without stop words and punctuation)
content = [t for t in art if not t.is_stop and not t.is_punct and not t.is_space and t.is_alpha]
top_words = Counter(t.text.lower() for t in content).most_common(10)
top_lemmas = Counter(t.lemma_.lower() for t in content).most_common(10)
pd.DataFrame({'word': [f'{w} ({c})' for w, c in top_words], 'lemma': [f'{w} ({c})' for w, c in top_lemmas]})

,word,lemma
0,pays (21),énergie (33)
1,gaz (20),être (24)
2,émissions (18),émission (21)
3,été (18),pays (21)
4,énergies (18),gaz (20)
5,publié (16),renouvelable (20)
6,ici (15),publier (17)
7,énergie (15),objectif (17)
8,renouvelables (15),ici (15)
9,serre (14),gouvernement (14)


In [9]:
# e) dependency parse of the shortest sentence with at least 8 tokens
from spacy import displacy
shortest = min((s for s in sents if len(s) >= 8 and '\n' not in s.text), key=len)
print(shortest.text)
displacy.render(shortest.as_doc(), style='dep', jupyter=True, options={'compact': True, 'distance': 95})

Du point de vue environnemental en premier lieu.


**Question 3.** The Guardian page could not be downloaded in the environment where this notebook was executed, so the outputs above use the **16 news texts of our French corpus** with the French model. When you run the notebook on your laptop, the same cells analyse the Guardian article.

Results on our news texts:
- **c)** 6,204 tokens and 198 sentences, i.e. **31.3 tokens per sentence**. The longest sentence is a long scenario description from the *100 % renouvelable en 2050* article.
- **d)** Words versus lemmas. The ranking changes because **lemmatization merges the inflected forms**:
  - *énergie* (15) and *énergies* (18) become the lemma *énergie* (33), which moves to **rank 1**;
  - *émissions* becomes *émission*;
  - the word *été* (18, the participle of *être*, not *summer*) disappears into the lemma *être* (24);
  - *publié* (from the date lines *Publié le …*) becomes *publier*.

  On the Guardian text you should see the same effect: *said / says → say*, *fans → fan*, etc.
- **e)** The shortest sentence with at least 8 tokens is *"Du point de vue environnemental en premier lieu."*: a sentence without a verb, which the parser attaches around the noun *point*.

---
# Part 3 — Stemming
## Question 4

In [10]:
families = {
    'argue':       ['argue', 'argued', 'argues', 'arguing', 'argument', 'arguments'],
    'analysis':    ['analysis', 'analyses', 'analyze', 'analyzed', 'analyzing', 'analytical'],
    'communicate': ['communicate', 'communicated', 'communication', 'communications'],
    'community':   ['community', 'communities'],
    'general':     ['general', 'generally', 'generalize', 'generalization'],
    'generate':    ['generate', 'generated', 'generation', 'generator'],
}
words = [w for ws in families.values() for w in ws]
family_of = {w: f for f, ws in families.items() for w in ws}
stemmers = {'Porter': PorterStemmer(), 'Snowball': SnowballStemmer('english'), 'Lancaster': LancasterStemmer()}
print(len(words), 'words')

# a) group the words by stem
def group_by_stem(words, stem):
    groups = defaultdict(list)
    for w in words:
        groups[stem(w)].append(w)
    return dict(groups)

for name, st in stemmers.items():
    print(f'\n{name}:')
    for stem, ws in group_by_stem(words, st.stem).items():
        print(f'   {stem:12s} {ws}')

26 words

Porter:
   argu         ['argue', 'argued', 'argues', 'arguing']
   argument     ['argument', 'arguments']
   analysi      ['analysis']
   analys       ['analyses']
   analyz       ['analyze', 'analyzed', 'analyzing']
   analyt       ['analytical']
   commun       ['communicate', 'communicated', 'communication', 'communications', 'community', 'communities']
   gener        ['general', 'generally', 'generalize', 'generalization', 'generate', 'generated', 'generation', 'generator']

Snowball:
   argu         ['argue', 'argued', 'argues', 'arguing']
   argument     ['argument', 'arguments']
   analysi      ['analysis']
   analys       ['analyses']
   analyz       ['analyze', 'analyzed', 'analyzing']
   analyt       ['analytical']
   communic     ['communicate', 'communicated', 'communication', 'communications']
   communiti    ['community', 'communities']
   general      ['general', 'generally', 'generalize', 'generalization']
   generat      ['generate', 'generated', 'generatio

In [11]:
# b) under-stemming (same family, different stems) and over-stemming (different families, same stem)
def stemming_errors(words, family_of, stem):
    under = over = same_pairs = diff_pairs = 0
    examples = {'under': [], 'over': []}
    for w1, w2 in combinations(words, 2):
        same_family, same_stem = family_of[w1] == family_of[w2], stem(w1) == stem(w2)
        if same_family:
            same_pairs += 1
            if not same_stem:
                under += 1; examples['under'].append((w1, w2))
        else:
            diff_pairs += 1
            if same_stem:
                over += 1; examples['over'].append((w1, w2))
    return {'under-stemming': under, 'under rate': round(under / same_pairs, 3),
            'over-stemming': over, 'over rate': round(over / diff_pairs, 3),
            'total errors': under + over, 'pairs same/diff family': f'{same_pairs}/{diff_pairs}'}, examples

rows, examples = {}, {}
for name, st in stemmers.items():
    rows[name], examples[name] = stemming_errors(words, family_of, st.stem)
pd.DataFrame(rows).T

,under-stemming,under rate,over-stemming,over rate,total errors,pairs same/diff family
Porter,20,0.408,24,0.087,44,49/276
Snowball,20,0.408,0,0.0,20,49/276
Lancaster,0,0.0,24,0.087,24,49/276


In [12]:
for name in stemmers:
    print(name, '| over-stemming:', examples[name]['over'][:6])
    print(' ' * len(name), '| under-stemming (first 6):', examples[name]['under'][:6])
# d) the suffixes that cause the surprising errors
for w in ['generate', 'general', 'generalize', 'communicate', 'community', 'analysis', 'analyze', 'argument']:
    print(f'{w:12s}', {n: s.stem(w) for n, s in stemmers.items()})

Porter | over-stemming: [('communicate', 'community'), ('communicate', 'communities'), ('communicated', 'community'), ('communicated', 'communities'), ('communication', 'community'), ('communication', 'communities')]
       | under-stemming (first 6): [('argue', 'argument'), ('argue', 'arguments'), ('argued', 'argument'), ('argued', 'arguments'), ('argues', 'argument'), ('argues', 'arguments')]
Snowball | over-stemming: []
         | under-stemming (first 6): [('argue', 'argument'), ('argue', 'arguments'), ('argued', 'argument'), ('argued', 'arguments'), ('argues', 'argument'), ('argues', 'arguments')]
Lancaster | over-stemming: [('communicate', 'community'), ('communicate', 'communities'), ('communicated', 'community'), ('communicated', 'communities'), ('communication', 'community'), ('communication', 'communities')]
          | under-stemming (first 6): []
generate     {'Porter': 'gener', 'Snowball': 'generat', 'Lancaster': 'gen'}
general      {'Porter': 'gener', 'Snowball': 'general

**Question 4: results.**

| Stemmer | Under-stemming | Rate | Over-stemming | Rate | Total |
|---|---|---|---|---|---|
| Porter | 20 | 0.41 | 24 | 0.087 | 44 |
| Snowball | 20 | 0.41 | **0** | 0 | **20** |
| Lancaster | **0** | 0 | 24 | 0.087 | 24 |

- **c)** **Snowball** makes the fewest errors (20, all under-stemming).
  - **Porter** makes both kinds: it splits the *argue* and *analysis* families, and merges *communicate*/*community* and *general*/*generate*.
  - **Lancaster** is the most aggressive: one stem per family (no under-stemming), but it also merges unrelated families (*gen* for *general* and *generate*, *commun* for *communicate* and *community*).
- **d) Three surprising errors:**
  1. ***general* and *generate* → *gener* (Porter).** Porter removes *-al* from *general* and *-ate* from *generate*: different suffixes, same remainder. Snowball avoids this on purpose: Porter2 has a special rule for words that start with *gener-*, *commun-* and *arsen-* (it does not cut inside these prefixes). That is why it keeps *general* and *generat*.
  2. ***communicate* and *community* → *commun* (Porter, Lancaster).** *communicate* loses *-icate* (step 3, then *-ic* in step 4). *community* becomes *communiti* and loses *-iti* in step 4.
  3. ***analysis / analyses / analyze / analytical* → four different stems (Porter, Snowball).** *-is* is not a suffix for Porter, *-es* is removed from *analyses*, and *-yze* / *-ytical* are not in its rules. The result is *analysi*, *analys*, *analyz*, *analyt*. *argue* and *argument* stay apart because *-ment* is only removed from longer stems.
- **e) Precision or recall?** Over-stemming merges unrelated words, so a search returns irrelevant documents and **precision drops**. Under-stemming separates related words, so relevant documents are missed and **recall drops**.
  - **For a search engine:** **Snowball**. Fewest errors, and no irrelevant matches. If recall matters most (find everything, even with noise), Lancaster.
  - **For a system that must not mix unrelated words:** **Snowball** (0 over-stemming), or better still lemmatization.

---
# Part 4 — Lemmatization
## f) WordNetLemmatizer with and without the part of speech

In [13]:
wnl = WordNetLemmatizer()
pairs = [('leaves', 'n'), ('leaves', 'v'), ('saw', 'n'), ('saw', 'v'), ('found', 'v'), ('founded', 'v'),
         ('lying', 'v'), ('feet', 'n'), ('wolves', 'n'), ('axes', 'n'), ('analyses', 'n'), ('dying', 'v'),
         ('went', 'v'), ('worse', 'a'), ('wounded', 'a'), ('indices', 'n')]
table_f = pd.DataFrame([{'word': w, 'pos': p, 'no pos (default noun)': wnl.lemmatize(w),
                         'with pos': wnl.lemmatize(w, pos=p)} for w, p in pairs])
table_f['default already correct'] = table_f['no pos (default noun)'] == table_f['with pos']
print('i) The default (noun) is already correct for', table_f['default already correct'].sum(), 'of', len(pairs), 'pairs')
table_f

i) The default (noun) is already correct for 9 of 16 pairs


,word,pos,no pos (default noun),with pos,default already correct
0,leaves,n,leaf,leaf,True
1,leaves,v,leaf,leave,False
2,saw,n,saw,saw,True
3,saw,v,saw,saw,True
4,found,v,found,find,False
5,founded,v,founded,found,False
6,lying,v,lying,lie,False
7,feet,n,foot,foot,True
8,wolves,n,wolf,wolf,True
9,axes,n,ax,ax,True


In [14]:
# ii) candidate lemmas from WordNet's morphology
for w, p in [('saw', 'v'), ('axes', 'n'), ('leaves', 'n')]:
    print(f'{w} ({p}): candidates {wn._morphy(w, p)} -> lemmatizer returns {wnl.lemmatize(w, p)!r}')

saw (v): candidates ['saw', 'see'] -> lemmatizer returns 'saw'
axes (n): candidates ['ax', 'axis'] -> lemmatizer returns 'ax'
leaves (n): candidates ['leaf', 'leave'] -> lemmatizer returns 'leaf'


**Question f: results.**

**i)** The default (noun) gives the same lemma as the correct part of speech for **9 of 16 pairs**: all the nouns (*leaves → leaf*, *feet → foot*, *wolves → wolf*, *indices → index*…) plus *wounded (a)* and *saw (v)*. But *saw (v) → saw* is **wrong in both cases**, so the default is really correct for 8 pairs. Every verb and the adjective *worse* need the POS (*went → go*, *worse → bad*, *dying → die*, *found → find*).

**ii)** WordNet returns the candidates *saw (v)* → [saw, see], *axes (n)* → [ax, axis] and *leaves (n)* → [leaf, leave]. The lemmatizer picks the shortest, or the first when they have the same length. This is right only by chance:

| Sentence | Correct lemma | Lemmatizer |
|---|---|---|
| He saw the film | **see** | saw ✘ |
| She cut the wood with a saw | **saw** | saw ✔ |
| The axes were found in the shed | **ax** | ax ✔ |
| The axes of the graph | **axis** | ax ✘ |

**iii)** *saw* as a verb can be the past tense of *see* or the present of *to saw* (to cut wood). Both are verbs, so the POS tag alone cannot decide. We also need the **tense** (a fine tag such as VBD versus VBP) and the **meaning in context** (word-sense disambiguation: is there wood or a plank?).

## g) Evaluation of two lemmatizers

In [15]:
text_g = ("He saw that the leaves were falling on the wolves, and the analyses showed worse results. "
          "She leaves the village while the leaves are falling. The geese flew over the old wolves.")
# i) gold lemmas, written by hand (lower-case, no punctuation)
gold_g = ('he see that the leaf be fall on the wolf and the analysis show bad result '
          'she leave the village while the leaf be fall the goose fly over the old wolf').split()

# ii) spaCy
sp_tokens = [t for t in nlp(text_g) if not t.is_punct]
sp_lemmas = [t.lemma_.lower() for t in sp_tokens]

# ii) NLTK: pos_tag (Penn Treebank tags) -> WordNet tags -> WordNetLemmatizer
def penn_to_wn(tag):
    return {'J': 'a', 'V': 'v', 'R': 'r', 'N': 'n'}.get(tag[0], 'n')
nltk_tagged = [(w, t) for w, t in nltk.pos_tag(word_tokenize(text_g)) if w.isalpha()]
nltk_lemmas = [wnl.lemmatize(w.lower(), penn_to_wn(t)) for w, t in nltk_tagged]

words_g = [w.lower() for w, _ in nltk_tagged]
assert len(words_g) == len(gold_g) == len(sp_lemmas)
acc = lambda pred: sum(p == g for p, g in zip(pred, gold_g)) / len(gold_g)
print(f'{len(gold_g)} words | accuracy spaCy = {acc(sp_lemmas):.3f} | accuracy NLTK = {acc(nltk_lemmas):.3f}')

# iii) the errors, with the POS tag that each tool used
err = []
for i, g in enumerate(gold_g):
    if sp_lemmas[i] != g or nltk_lemmas[i] != g:
        err.append({'word': words_g[i], 'gold': g, 'spaCy': sp_lemmas[i], 'spaCy POS': sp_tokens[i].tag_,
                    'NLTK': nltk_lemmas[i], 'NLTK POS': nltk_tagged[i][1]})
pd.DataFrame(err)

32 words | accuracy spaCy = 0.969 | accuracy NLTK = 0.906


,word,gold,spaCy,spaCy POS,NLTK,NLTK POS
0,saw,see,see,VBD,saw,VBD
1,geese,goose,geese,NN,geese,JJ
2,flew,fly,fly,VBD,flew,NN


In [16]:
# iv) "The workers found the axes in the shed."
s4 = 'The workers found the axes in the shed.'
print('spaCy:', [(t.text, t.lemma_) for t in nlp(s4) if t.text in ('found', 'axes')])
print('NLTK :', [(w, wnl.lemmatize(w, penn_to_wn(t))) for w, t in nltk.pos_tag(word_tokenize(s4)) if w in ('found', 'axes')])
print('WordNet candidates for axes (n):', wn._morphy('axes', 'n'))

spaCy: [('found', 'find'), ('axes', 'axis')]
NLTK : [('found', 'find'), ('axes', 'ax')]
WordNet candidates for axes (n): ['ax', 'axis']


**Question g: results.** Accuracy on the 32 words: **spaCy 0.969** (1 error), **NLTK 0.906** (3 errors).

| Word | Gold | spaCy | NLTK | Cause of the NLTK error |
|---|---|---|---|---|
| saw | see | see ✔ | saw ✘ | **lemmatizer**: the tag VBD is correct, but WordNet returns the first candidate *saw* |
| geese | goose | geese ✘ | geese ✘ | **tagger**: NLTK tags it JJ (adjective) |
| flew | fly | fly ✔ | flew ✘ | **tagger**: NLTK tags it NN (noun) |

The two *leaves* (noun and verb) in *"She leaves the village while the leaves are falling"* are lemmatized correctly by **both** tools: the taggers got the context right.

**iv)** In *"The workers found the axes in the shed"* the correct lemma is ***ax*** (tools in a shed). spaCy returns ***axis*** (wrong): it picks the lemma it learned as more likely. NLTK returns ***ax*** (right), but only because of its "shortest candidate" rule, not because it understood the sentence.

**v)** In a pipeline we would use **spaCy**. It is more accurate, it uses the context, and it tags and lemmatizes in one step, whereas with NLTK every tagger error becomes a lemma error. NLTK is useful when we already know the POS or need to stay very light.

## h) Stemming or lemmatization?

In [17]:
pairs_h = [('flies', 'v'), ('caring', 'v'), ('argument', 'n'), ('analyses', 'n'), ('feet', 'n'),
           ('wives', 'n'), ('happier', 'a'), ('nationalization', 'n')]
spacy_pos = {'v': 'VERB', 'n': 'NOUN', 'a': 'ADJ'}

def spacy_lemma(word, pos):
    # spaCy lemmatizes in context, so we give each word a small context with the right part of speech
    frame = {'v': 'It {} .', 'n': 'The {} .', 'a': 'It is {} .'}[pos].format(word)
    return next(t.lemma_ for t in nlp(frame) if t.text == word)

table_h = pd.DataFrame([{'word': w, 'pos': p,
    **{n: s.stem(w) for n, s in stemmers.items()},
    'spaCy lemma': spacy_lemma(w, p), 'NLTK lemma': wnl.lemmatize(w, p)} for w, p in pairs_h])
table_h['type'] = ['regular inflection', 'regular inflection', 'derivation', 'irregular inflection',
                   'irregular inflection', 'irregular inflection', 'regular inflection (comparative)', 'derivation']
table_h

,word,pos,Porter,Snowball,Lancaster,spaCy lemma,NLTK lemma,type
0,flies,v,fli,fli,fli,fly,fly,regular inflection
1,caring,v,care,care,car,care,care,regular inflection
2,argument,n,argument,argument,argu,argument,argument,derivation
3,analyses,n,analys,analys,analys,analysis,analysis,irregular inflection
4,feet,n,feet,feet,feet,foot,foot,irregular inflection
5,wives,n,wive,wive,wiv,wife,wife,irregular inflection
6,happier,a,happier,happier,happy,happy,happy,regular inflection (comparative)
7,nationalization,n,nation,nation,nat,nationalization,nationalization,derivation


In [18]:
# iii) speed: 3 stemmers and WordNetLemmatizer (pos='n') on all alphabetic tokens of the article, then nlp()
alpha = [t.text.lower() for t in art if t.is_alpha]
timings = {}
for name, st in stemmers.items():
    t0 = time.perf_counter(); [st.stem(w) for w in alpha]; timings[name] = time.perf_counter() - t0
t0 = time.perf_counter(); [wnl.lemmatize(w, 'n') for w in alpha]; timings['WordNetLemmatizer'] = time.perf_counter() - t0
t0 = time.perf_counter(); nlp_art(article_text); timings['spaCy nlp(text)'] = time.perf_counter() - t0
print(len(alpha), 'alphabetic tokens of', SOURCE)
pd.DataFrame({'seconds': timings}).assign(ms_per_1000_tokens=lambda d: (1000 * d.seconds / len(alpha) * 1000).round(2)).round(4)

4789 alphabetic tokens of the 16 news texts of our French corpus


,seconds,ms_per_1000_tokens
Porter,0.0296,6.19
Snowball,0.0150,3.12
Lancaster,0.1968,41.10
WordNetLemmatizer,0.0167,3.48
spaCy nlp(text),0.7372,153.93


**Question h: results.**

**ii)** Classification of the 8 words:
- **regular inflection:** *flies, caring, happier*
- **irregular inflection:** *analyses, feet, wives*
- **derivation:** *argument* (from *argue*), *nationalization* (from *nation → national → nationalize*)

For each word, the best method:
- **Search engine:** stems are enough for regular inflection (*fli*, *care*). But they fail on irregular forms: *feet* stays *feet*, *wives → wive*, and *analyses → analys* while *analysis → analysi*. **Lemmas** are better there. For derivation, stemming merges more (*nationalization → nation*), which raises recall but can mix meanings (a *nation* is not a *nationalization*).
- **Text for people to read:** always **lemmas**, because stems such as *fli*, *wive* or *nat* (Lancaster) are not words.

**iii) Speed** (in our run, on the 4,789 alphabetic tokens of the text used):
- Snowball about 3 ms per 1,000 tokens, Porter about 6, Lancaster about 37 (it applies its rules again and again);
- WordNetLemmatizer about 3 ms (fast here because we give it `pos='n'` and it only looks words up);
- the **full spaCy pipeline** about 130 ms, i.e. about **40 times slower** than Snowball.

This agrees with the course's "stemming fast / lemmatization slow", with one nuance: the lookup itself is fast. What is slow is the analysis that lemmatization needs (tagging, parsing).

---
# Part 5 — WordNet
## i) Synsets and senses

In [19]:
syn = wn.synsets('plant')
print(len(syn), 'synsets of "plant":', Counter(s.pos() for s in syn))
for s in syn[:3]:
    print(f'  {s.name():15s} {s.definition()} | lemmas: {s.lemma_names()}')
c = wn.synset('computer.n.01')
print('\ncomputer.n.01:', c.lemma_names(), '|', c.definition())

10 synsets of "plant": Counter({'v': 6, 'n': 4})
  plant.n.01      buildings for carrying on industrial labor | lemmas: ['plant', 'works', 'industrial_plant']
  plant.n.02      (botany) a living organism lacking the power of locomotion | lemmas: ['plant', 'flora', 'plant_life']
  plant.n.03      an actor situated in the audience whose acting is rehearsed but seems spontaneous to the audience | lemmas: ['plant']

computer.n.01: ['computer', 'computing_machine', 'computing_device', 'data_processor', 'electronic_computer', 'information_processing_system'] | a machine for performing calculations automatically


## j) Hypernyms and hyponyms

In [20]:
print('Hypernyms of computer.n.01:', c.hypernyms())
print('First five hyponyms       :', c.hyponyms()[:5])
apple = wn.synset('apple.n.01')
paths = apple.hypernym_paths()
print(f'\napple.n.01 has {len(paths)} hypernym paths:')
for p in paths:
    print(f'   {len(p) - 1} steps:', ' -> '.join(s.name() for s in reversed(p)))
print('\nPart meronyms of tree.n.01:', wn.synset('tree.n.01').part_meronyms()[:5])
print('Antonyms of the lemma hot :', [a.name() for l in wn.synsets('hot', 'a') for lm in l.lemmas() if lm.name() == 'hot' for a in lm.antonyms()])
print('Entailments of buy.v.01   :', wn.synset('buy.v.01').entailments())

Hypernyms of computer.n.01: [Synset('machine.n.01')]
First five hyponyms       : [Synset('node.n.08'), Synset('digital_computer.n.01'), Synset('web_site.n.01'), Synset('server.n.03'), Synset('number_cruncher.n.02')]

apple.n.01 has 3 hypernym paths:
   10 steps: apple.n.01 -> pome.n.01 -> fruit.n.01 -> reproductive_structure.n.01 -> plant_organ.n.01 -> plant_part.n.01 -> natural_object.n.01 -> whole.n.02 -> object.n.01 -> physical_entity.n.01 -> entity.n.01
   10 steps: apple.n.01 -> edible_fruit.n.01 -> fruit.n.01 -> reproductive_structure.n.01 -> plant_organ.n.01 -> plant_part.n.01 -> natural_object.n.01 -> whole.n.02 -> object.n.01 -> physical_entity.n.01 -> entity.n.01
   7 steps: apple.n.01 -> edible_fruit.n.01 -> produce.n.01 -> food.n.02 -> solid.n.01 -> matter.n.03 -> physical_entity.n.01 -> entity.n.01

Part meronyms of tree.n.01: [Synset('trunk.n.01'), Synset('limb.n.02'), Synset('stump.n.01'), Synset('crown.n.07'), Synset('burl.n.02')]
Antonyms of the lemma hot : ['cold', 'c

## k) Similarity

In [21]:
coffee, tea, chair = wn.synset('coffee.n.01'), wn.synset('tea.n.01'), wn.synset('chair.n.01')
for x, y in [(coffee, tea), (coffee, chair)]:
    print(f'{x.name()} - {y.name()}: path = {x.path_similarity(y):.3f}, wup = {x.wup_similarity(y):.3f}, '
          f'lowest common hypernym = {x.lowest_common_hypernyms(y)}')
n = coffee.shortest_path_distance(tea)
print(f'\niii) by hand: shortest path coffee-tea = {n} edges -> 1 / (1 + {n}) = {1 / (1 + n):.3f}')

coffee.n.01 - tea.n.01: path = 0.333, wup = 0.889, lowest common hypernym = [Synset('beverage.n.01')]
coffee.n.01 - chair.n.01: path = 0.071, wup = 0.235, lowest common hypernym = [Synset('physical_entity.n.01')]

iii) by hand: shortest path coffee-tea = 2 edges -> 1 / (1 + 2) = 0.333


**WordNet results.**
- **i)** *plant* has **10 synsets: 4 nouns and 6 verbs**.
  - *plant.n.01* is a factory, with the synonyms *plant, works, industrial_plant*;
  - *plant.n.02* is the living organism (*plant, flora, plant_life*);
  - *plant.n.03* is an actor planted in the audience.

  *computer.n.01* has 6 lemmas (*computer, computing_machine, data_processor…*) and the definition "a machine for performing calculations automatically".
- **j)**
  - *computer.n.01* **is a** *machine.n.01*.
  - *apple.n.01* has **3 hypernym paths** to *entity.n.01*, of **10, 7 and 10 steps**. There are several paths because WordNet is a graph, not a tree: an apple is both an *edible fruit* (food, produce, matter) and a botanical *fruit* / *pome* (a plant part).
  - The parts of a tree: *stump, crown, burl, limb, trunk*. The antonym of *hot*: *cold*. Buying entails *paying* and *choosing*.
- **k)**
  - *coffee–tea*: path 0.333, Wu–Palmer 0.889, common hypernym ***beverage***.
  - *coffee–chair*: path 0.071, Wu–Palmer 0.235, common hypernym *physical_entity*.
  - **Check by hand:** coffee → beverage → tea is 2 edges, so 1/(1 + 2) = **0.333**. ✔

  Coffee and tea are clearly the more similar pair, which agrees with intuition.

---
# Part 6 — The same ideas on our French corpus (project)
The lab is in English, but our project is in French. French has **more inflection** than English: one noun can have 2 forms (*émission, émissions*), an adjective 4 (*climatique, climatiques*, *renouvelable, renouvelables*; *polluant, polluante, polluants, polluantes*) and a verb dozens (*polluer, pollue, polluent, pollué, polluait…*).

We compare the **Snowball French stemmer** (NLTK) with the **spaCy French lemmatizer** (`fr_core_news_md`) in two ways:
1. on **French word families** grouped by hand (same method as question 4);
2. on the **vocabulary of our corpus**: how many index terms does each method leave?

In [22]:
from nltk.stem.snowball import FrenchStemmer
fr_stem = FrenchStemmer()
nlp_fr = spacy.load('fr_core_news_md')

fr_families = {
    'polluer':     ['pollution', 'pollutions', 'polluer', 'pollue', 'polluent', 'pollué', 'polluée', 'polluant', 'polluants', 'pollueur'],
    'énergie':     ['énergie', 'énergies', 'énergétique', 'énergétiques'],
    'émettre':     ['émission', 'émissions', 'émettre', 'émet', 'émis', 'émise', 'émetteur', 'émetteurs'],
    'climat':      ['climat', 'climats', 'climatique', 'climatiques'],
    'renouveler':  ['renouvelable', 'renouvelables', 'renouveler', 'renouvellement'],
    'carbone':     ['carbone', 'carbones', 'carbonique', 'carbonisation'],
    'organe':      ['organe', 'organes'],
    'organiser':   ['organisation', 'organisations', 'organiser', 'organisé'],
    'général':     ['général', 'générale', 'généralement', 'généraliser'],
    'générer':     ['générer', 'génère', 'généré', 'générateur'],
}
fr_words = [w for ws in fr_families.values() for w in ws]
fr_family_of = {w: f for f, ws in fr_families.items() for w in ws}

def spacy_fr_lemma(w):
    return nlp_fr(w)[0].lemma_.lower()      # a word without context: the lemmatizer has to guess the POS

methods = {'Snowball (French)': fr_stem.stem, 'spaCy lemma': spacy_fr_lemma}
pd.DataFrame({name: stemming_errors(fr_words, fr_family_of, f)[0] for name, f in methods.items()}).T

,under-stemming,under rate,over-stemming,over rate,total errors,pairs same/diff family
Snowball (French),64,0.552,0,0.0,64,116/1012
spaCy lemma,90,0.776,0,0.0,90,116/1012


In [23]:
for name, f in methods.items():
    print(f'\n{name}:')
    for key, ws in group_by_stem(fr_words, f).items():
        fams = {fr_family_of[w] for w in ws}
        flag = '   <-- over: several families' if len(fams) > 1 else ''
        print(f'   {key:16s} {ws}{flag}')


Snowball (French):
   pollut           ['pollution', 'pollutions']
   pollu            ['polluer', 'pollue', 'pollué', 'polluée', 'polluant', 'polluants']
   polluent         ['polluent']
   pollueur         ['pollueur']
   énerg            ['énergie', 'énergies']
   énerget          ['énergétique', 'énergétiques']
   émiss            ['émission', 'émissions']
   émettr           ['émettre']
   émet             ['émet']
   émis             ['émis', 'émise']
   émetteur         ['émetteur', 'émetteurs']
   climat           ['climat', 'climats', 'climatique', 'climatiques']
   renouvel         ['renouvelable', 'renouvelables', 'renouveler', 'renouvellement']
   carbon           ['carbone', 'carbones', 'carbonique']
   carbonis         ['carbonisation']
   organ            ['organe', 'organes']
   organis          ['organisation', 'organisations', 'organiser', 'organisé']
   général          ['général', 'générale', 'généralement']
   généralis        ['généraliser']
   géner            [

   pollution        ['pollution', 'pollutions']
   polluer          ['polluer', 'polluent', 'pollué']
   pollu            ['pollue']
   polluée          ['polluée']
   polluant         ['polluant', 'polluants']
   pollueur         ['pollueur']
   énergie          ['énergie', 'énergies']
   énergétique      ['énergétique', 'énergétiques']
   émission         ['émission', 'émissions']
   émettre          ['émettre', 'émet', 'émis', 'émise']
   émetteur         ['émetteur', 'émetteurs']
   climat           ['climat', 'climats']
   climatique       ['climatique', 'climatiques']
   renouvelable     ['renouvelable', 'renouvelables']
   renouveler       ['renouveler']
   renouvellement   ['renouvellement']
   carbone          ['carbone', 'carbones']
   carbonique       ['carbonique']
   carbonisation    ['carbonisation']
   organe           ['organe', 'organes']
   organisation     ['organisation', 'organisations']
   organiser        ['organiser', 'organisé']
   général          ['général', 

In [24]:
# The vocabulary of our corpus: how many index terms with each normalization?
from nlp_utils import load_corpus
docs = load_corpus()
fr_docs = list(nlp_fr.pipe([d['text'] for d in docs], disable=['ner']))
alpha_fr = [t for d in fr_docs for t in d if t.is_alpha]

vocab = {
    'word forms (lower-case)': {t.lower_ for t in alpha_fr},
    'Snowball stems':          {fr_stem.stem(t.lower_) for t in alpha_fr},
    'spaCy lemmas':            {t.lemma_.lower() for t in alpha_fr},
}
base = len(vocab['word forms (lower-case)'])
print(f'{len(alpha_fr)} alphabetic tokens')
for k, v in vocab.items():
    print(f'   {k:26s} {len(v):5d} types ({100 * len(v) / base:.0f} %)')

# Which forms are merged? The largest groups for each method
for name, key in [('Snowball', lambda t: fr_stem.stem(t.lower_)), ('spaCy', lambda t: t.lemma_.lower())]:
    groups = defaultdict(set)
    for t in alpha_fr:
        if not t.is_stop:
            groups[key(t)].add(t.lower_)
    big = sorted(groups.items(), key=lambda kv: -len(kv[1]))[:6]
    print(f'\n{name}: largest groups')
    for k, forms in big:
        print(f'   {k:14s} {sorted(forms)}')

25882 alphabetic tokens
   word forms (lower-case)     4890 types (100 %)
   Snowball stems              3266 types (67 %)
   spaCy lemmas                3543 types (72 %)

Snowball: largest groups
   présent        ['présent', 'présentant', 'présentation', 'présente', 'présenter', 'présentes', 'présents', 'présenté', 'présentée', 'présentées', 'présentés']
   consomm        ['consommant', 'consommateur', 'consommateurs', 'consommation', 'consommations', 'consommer', 'consommé', 'consommée', 'consommées', 'consommés']
   import         ['importance', 'important', 'importante', 'importantes', 'importants', 'importateurs', 'importation', 'importations', 'importé', 'importés']
   utilis         ['utilisable', 'utilisant', 'utilisation', 'utilise', 'utiliser', 'utilisé', 'utilisée', 'utilisées', 'utilisés']
   pollu          ['polluant', 'polluante', 'polluantes', 'polluants', 'polluer', 'pollué', 'polluées', 'pollués']
   limit          ['limitation', 'limite', 'limiter', 'limites', 'limi

In [25]:
# Speed on French: Snowball vs the full spaCy pipeline, on our whole corpus
words_fr = [t.lower_ for t in alpha_fr]
t0 = time.perf_counter(); [fr_stem.stem(w) for w in words_fr]; t_stem = time.perf_counter() - t0
t0 = time.perf_counter(); list(nlp_fr.pipe([d['text'] for d in docs], disable=['ner'])); t_spacy = time.perf_counter() - t0
print(f'{len(words_fr)} tokens: Snowball {t_stem:.2f} s, spaCy pipeline {t_spacy:.2f} s ({t_spacy / t_stem:.0f} times slower)')

25882 tokens: Snowball 0.16 s, spaCy pipeline 2.14 s (13 times slower)


## WordNet in French
WordNet itself is English, but the **Open Multilingual WordNet** (included in NLTK) links French words to the same synsets (`lang='fra'`).

In [26]:
for w in ['pollution', 'énergie', 'carbone', 'climat', 'éolienne']:
    ss = wn.synsets(w, lang='fra')
    print(f'{w:10s} {len(ss)} synsets:', [(s.name(), s.lemma_names('fra')[:4]) for s in ss[:3]])

pol = wn.synset('pollution.n.01')
print('\npollution.n.01 :', pol.definition())
print('French lemmas  :', pol.lemma_names('fra'))
print('Hypernym path  :', ' -> '.join(f"{s.name()} {s.lemma_names('fra')[:1]}" for s in reversed(pol.hypernym_paths()[0])))
for x, y in [('coal.n.01', 'petroleum.n.01'), ('coal.n.01', 'wind.n.01'), ('pollution.n.01', 'contamination.n.01')]:
    sx, sy = wn.synset(x), wn.synset(y)
    print(f'{x:20s} {y:20s} path={sx.path_similarity(sy):.3f}  wup={sx.wup_similarity(sy):.3f}  '
          f"({sx.lemma_names('fra')[:1]} / {sy.lemma_names('fra')[:1]})")

pollution  3 synsets: [('contamination.n.03', ['contamination', 'pollution']), ('befoulment.n.01', ['nuisance', 'polluant', 'pollution']), ('pollution.n.01', ['pollution'])]
énergie    6 synsets: [('energy.n.04', ['énergie']), ('energy.n.03', ['énergie', 'pousser']), ('energy.n.02', ['énergétique', 'énergie', 'force', 'vigueur'])]
carbone    4 synsets: [('carbon.n.03', ['carbone', 'charbon', 'copie', 'copie_carbone']), ('discharge.n.04', ['carbone', 'échappement', 'émission', 'polluant']), ('carbon.n.01', ['carbone', 'charbon'])]
climat     2 synsets: [('climate.n.01', ['climat', 'convention-cadre']), ('climate.n.02', ['climat', 'convention-cadre', 'environnement', 'humeur'])]
éolienne   3 synsets: [('windmill.n.01', ['éolienne', 'moulin_à_vent']), ('wind_turbine.n.01', ['éolienne', 'Éolienne']), ('wind.n.04', ['éolienne', 'jazz', 'néant', 'vent'])]

pollution.n.01 : undesirable state of the natural environment being contaminated with harmful substances as a consequence of human activi

**What this means for our French project.**

**1. French word families.** Both methods make **no over-stemming** errors, unlike Porter in English:
- *organe* and *organiser* stay apart, and so do *général* and *générer*.

Both make a lot of "under-stemming" by our family definition:
- **Snowball** (64 errors) merges *climat + climatique*, *renouvelable + renouveler + renouvellement*, and *pollué + polluant + polluer*. But it separates *pollution* (*pollut*) from *polluer* (*pollu*), *énergie* from *énergétique*, and *émission* from *émettre* / *émis*. French changes the stem there, and stemmers cannot follow.
- **spaCy lemmas** (90 errors) keep **derivations** apart **by design** (*climat ≠ climatique*, *pollution ≠ polluer*). As the lecture says, lemmatization handles inflection, not derivation. In exchange, spaCy merges irregular verb forms that Snowball never can (*émet, émis, émise → émettre*).
- Words given **without context** are sometimes lemmatized strangely (*pollue → pollu*, *polluée* unchanged). In real sentences the tagger has context and does better.

**2. The vocabulary of our corpus** (25,882 tokens):

| Normalization | Types | Share of the word forms |
|---|---|---|
| word forms | 4,890 | 100 % |
| Snowball stems | 3,266 | 67 % |
| spaCy lemmas | 3,543 | 72 % |

So the index shrinks by a third. The biggest groups show the trade-off:
- Snowball also **over-merges** real words: *import* covers *important* and *importation*, and *présent* covers *présentation* and *présenter*.
- spaCy merges whole conjugations correctly: *pourra, puisse, pouvaient → pouvoir*.

**3. Speed.** Snowball handles the whole corpus in about 0.15 s, the spaCy pipeline in about 1.6 s. Both are fine at our scale.

**4. French WordNet (OMW).** It works (*charbon–pétrole* Wu–Palmer 0.86 versus *charbon–vent* 0.29) but it is **noisy**: *climat* is linked to *convention-cadre* and *humeur*, and *éolienne* to *jazz* and *néant* (through the English *wind*). The French words were aligned automatically with the English synsets.

**Conclusion for the search engine.** Normalization clearly helps: fewer index terms, more matches. Stems and lemmas each have their errors, so in the project notebook we **measure** which gives the better search results (Precision@5) instead of guessing.